![IITIS](pictures/logoIITISduze.png)

# Wykorzystanie procesorów graficznych (GPU) w algorytmach heurystycznych

wykorzystamy paczkę numba (można też użyć CuPy wy bezpośrednio użwyać CUDA)

In [ ]:
# funkcje pomocniczne

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

test_pegasus = os.path.join("instancje", "Pegasus", "P4_CBFM-P.txt")  # E = -469.0
full_pegasus = os.path.join("instancje", "Pegasus", "P16_CBFM-P.txt")  # E = -12772.0



def read_instance(path: os.PathLike):
    df = pd.read_csv(path, sep=" ", header=None, comment="#", names=["i", "j", "value"])

    n = max(df[["i", "j"]].max())
    h = np.zeros(n)
    J = np.zeros((n, n))
    
    for row in df.itertuples():
        if row.i == row.j:
            h[row.i - 1] = row.value
        elif row.i > row.j:
            J[row.j - 1, row.i - 1] = row.value  # by zachować górnotrójkątność
        else:
            J[row.i - 1, row.j - 1] = row.value
            
    return J, h



In [1]:
# Parrarel annealing
# konwencja: J górnotrójkątna, plusy

import numpy as np
from typing import Optional
from tqdm import tqdm

def calculate_gradient_pararell(J: np.ndarray, h: np.ndarray, x: np.ndarray, state: np.ndarray, 
lambda_t: float) -> np.ndarray:
    _, num_trajectories = x.shape
    return J @ state + J.T @ state + np.stack([h] * num_trajectories, axis=1) + lambda_t * x


def calculate_energy_parrarel(J: np.ndarray, h: np.ndarray, state: np.ndarray, convention: str = "PLUS"):
    n, _ = J.shape
    if convention == "PLUS":
        # Zakładamy, że J jest górnotrójkątna z zerami na przekątnych
        return np.sum(state * ( J @ state + h.reshape(n, 1)), axis=0)
    else:
        return np.sum(state * ( -1/2 * J @ state - h.reshape(n, 1)), axis=0)


def get_best_solution(states, energies):
    best_energy = np.min(energies)
    best_energy_index = np.argmin(energies)
    best_state = states[:, best_energy_index]  # Stany są w kolumnach
    return best_state, best_energy


def parrarel_annealing_multiple_trajectores(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[np.ndarray] = None):
    n = len(h)
    x = np.zeros((n, num_trajectories))  # stan podstawowy dla H_innit = sum(x**2)
    momentum = np.zeros((n, num_trajectories))
    state = np.random.choice([-1, 1], size=(n, num_trajectories))  # losowy stan początkowy

    if schedule is None:
        schedule = np.linspace(lambda_t_max, 0, num=num_steps)

    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe"):
        lambda_t = schedule[k]
        gradient = calculate_gradient_pararell(J, h, x, state, lambda_t)
        momentum = (1 - step_size) * momentum - step_size * gradient
        momentum = np.clip(momentum, -1, 1)
        x += momentum
        x = np.clip(x, -1, 1)
        state = np.sign(x)

    return state, calculate_energy_parrarel(J, h, state, num_trajectories)

In [5]:
import cupy as cp
from numba import cuda


@cuda.jit
def increment_by_one(an_array):
    # Thread id in a 1D block
    tx = cuda.threadIdx.x
    # Block id in a 1D grid
    ty = cuda.blockIdx.x
    # Block width, i.e. number of threads per block
    bw = cuda.blockDim.x
    # Compute flattened index inside the array
    pos = tx + ty * bw
    if pos < an_array.size:  # Check array boundaries
        an_array[pos] += 1

an_array = cp.random.random(10000)

threadsperblock = 32
blockspergrid = (an_array.size + (threadsperblock - 1)) // threadsperblock
increment_by_one[blockspergrid, threadsperblock](an_array)



In [35]:
import cupy as cp

N = 1000

a = cp.random.random((N, N))
b = cp.random.random((N, N))
c = cp.matmul(a, b)
c.device

<CUDA Device 0>

In [69]:
# naiwna implementacja

def calculate_gradient_gpu_naive(J, h, x, state, lambda_t):
    _, num_trajectories = state.shape
    gradient = cp.matmul(J, state) + cp.matmul(J.T, state) + h + cp.multiply(lambda_t, x)
    print(gradient.device)
    return gradient


def calculate_energy_gpu_naive(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    n, _ = J.shape
    # Zakładamy, że J jest górnotrójkątna z zerami na przekątnych
    A = cp.matmul(J, state) + h.reshape(n, 1)
    print(A.device)
    B = cp.multiply(state, A)
    print(B.device)
    return cp.sum(B, axis=0)


    #return cp.sum(state * ( J @ state + h.reshape(n, 1)), axis=0)

N = 100
M = 32

J = cp.random.random((N, N))
h = cp.random.random(N)
h_parrarel = cp.stack([h] * M, axis=1)
x = cp.random.random((N, M))
state = cp.random.choice([-1, 1], size=(N, M))

calculate_energy_gpu_naive(J, h, state)

<CUDA Device 0>
<CUDA Device 0>


array([-16.49602537, -15.60906896, -49.96448297,  11.83746014,
        56.08271797,  64.66647897, 131.05826601,  60.38293547,
       -50.35029444,   6.79091073,  -0.35268449,  81.8859805 ,
       193.57908364,  14.24448576, -22.940412  ,  85.95132916,
        24.46258476, -15.98632205,  66.42098748,  73.30625326,
        35.76137584, 129.95455032,  40.53694691,   1.90733351,
         7.26332937,  95.36875517,  32.83288781,  -6.24808307,
        -2.06941621,  36.17591967,  16.31254888,  31.62345278])

In [ ]:
def calculate_gradient_gpu_naive(J, h, x, state, lambda_t):
    gradient = cp.matmul(J, state) + cp.matmul(J.T, state) + h + cp.multiply(lambda_t, x)
    return gradient


def calculate_energy_gpu_naive(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    n, _ = J.shape
    # Zakładamy, że J jest górnotrójkątna z zerami na przekątnych
    A = cp.matmul(J, state) + h.reshape(n, 1)
    B = cp.multiply(state, A)
    return cp.sum(B, axis=0)



def parrarel_annealing_gpu_naive(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[np.ndarray] = None, dtype = None):
    n = len(h)
    x = cp.zeros((n, num_trajectories))  # stan podstawowy dla H_innit = sum(x**2)
    momentum = cp.zeros((n, num_trajectories))
    state = cp.random.choice([-1, 1], size=(n, num_trajectories))  # losowy stan początkowy

    h_parrarel = cp.stack([h] * num_trajectories, axis=1)
    beta = cp.float32(1 - step_size)

    if schedule is None:
        schedule = cp.linspace(lambda_t_max, 0, num=num_steps)

    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe GPU"):
        lambda_t = schedule[k]
        gradient = calculate_gradient_gpu_naive(J, h_parrarel, x, state, lambda_t)
        momentum = cp.multiply(beta, momentum) - cp.multiply(step_size, gradient)
        momentum = cp.clip(momentum, -1, 1)
        x = cp.clip(x + momentum, -1, 1)  
        state = cp.sign(x)


    return state, calculate_energy_gpu_naive(J, h, state)


J, h = read_instance(test_pegasus)
J = cp.asarray(J)
h = cp.asarray(h)

states, energies = parrarel_annealing_gpu_naive(J, h, step_size=0.01, lambda_t_max=10, num_steps=1000, num_trajectories=32)
print(min(energies))

wyżarzanie równoległe GPU:   0%|          | 0/1000 [00:00<?, ?it/s]


ValueError: operands could not be broadcast together with shapes (216, 32) (32, 216)

In [76]:
# test na pełnym pegazie
J, h = read_instance(full_pegasus)
J = cp.asarray(J)
h = cp.asarray(h)

states, energies = parrarel_annealing_gpu_naive(J, h, step_size=0.01, lambda_t_max=10, num_steps=10000, num_trajectories=2**10)
print(min(energies))

wyżarzanie równoległe GPU:   3%|▎         | 298/10000 [00:49<26:50,  6.02it/s]


KeyboardInterrupt: 

In [34]:
# własny kernel

import numpy as np
from numba import cuda

num_trajectories = 32

J, h = read_instance(test_pegasus)
N = len(h)
x = np.zeros((N, num_trajectories))  
momentum = np.zeros((N, num_trajectories))
state = np.random.choice([-1, 1], size=(N, num_trajectories))

# move to GPU

J = cuda.to_device(J)


wyżarzanie równoległe:   0%|          | 0/1000 [00:00<?, ?it/s]

wyżarzanie równoległe: 100%|██████████| 1000/1000 [00:02<00:00, 334.26it/s]

228.0


# 